In [1]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

In [3]:
project_root = Path().resolve().parent

df = pd.read_csv(project_root / 'data' / 'processed' / 'node_combined_preprocess.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

In [4]:
## pisahin node menjadi per df

df1 = df[df['node'] == 1].copy()
df2 = df[df['node'] == 2].copy()
df3 = df[df['node'] == 3].copy()
df4 = df[df['node'] == 4].copy()
df5 = df[df['node'] == 5].copy()
df6 = df[df['node'] == 6].copy()


In [5]:
## Feature Engineering

def apply_feature_engineering(df_node):
    df_node["datetime"] = pd.to_datetime(df_node["datetime"], errors="coerce")

    df_node["hour"] = df_node["datetime"].dt.hour
    df_node["minute"] = df_node["datetime"].dt.minute
    df_node["minute_of_day"] = df_node["hour"] * 60 + df_node["minute"]

    df_node["h2s_diff"] = (
        df_node.groupby("node", observed=False)["h2s"]
        .diff()
        .fillna(0)
    )

    df_node["so2_diff"] = (
        df_node.groupby("node", observed=False)["so2"]
        .diff()
        .fillna(0)
    )

    df_node["gas_ratio_so2_h2s"] = np.where(
        df_node["h2s"] == 0,
        0,
        df_node["so2"] / (df_node["h2s"] + 1e-6)
    )
    return df_node

# Apply feature engineering to each node dataframe
for i in range(1, 7):
    df_name = f"df{i}"
    if df_name not in globals():
        print(f"Warning: {df_name} tidak ada. Skipping feature engineering untuk node ini.")
        continue

    #print(f"\Melakukan feature engineering untuk {df_name} (Node {i})...")
    globals()[df_name] = apply_feature_engineering(globals()[df_name].copy())
    print(f"{df_name} head setelah engineering:\n")
    print(globals()[df_name].head())

df1 head setelah engineering:

   node         location weather            datetime   h2s    so2    hum  \
0     1  Dekat uap panas   Cerah 2025-12-22 11:53:00  43.0  149.0  87.03   
1     1  Dekat uap panas   Cerah 2025-12-22 11:53:02  40.0  149.0  87.03   
2     1  Dekat uap panas   Cerah 2025-12-22 11:53:04  40.0  149.0  87.23   
3     1  Dekat uap panas   Cerah 2025-12-22 11:53:06  40.0  141.0  87.23   
4     1  Dekat uap panas   Cerah 2025-12-22 11:53:08  40.0  133.0  87.23   

    temp  windspeed  elevation  latitude   longitude  hour  minute  \
0  17.94       1.44     2101.0  -7.16687  107.401387    11      53   
1  17.94       0.72     2101.0  -7.16687  107.401387    11      53   
2  17.94       0.72     2101.0  -7.16687  107.401387    11      53   
3  17.94       0.72     2101.0  -7.16687  107.401387    11      53   
4  17.93       0.72     2101.0  -7.16687  107.401387    11      53   

   minute_of_day  h2s_diff  so2_diff  gas_ratio_so2_h2s  
0            713       0.0       

In [6]:
list_df = [df1, df2, df3, df4, df5, df6]
df_combined = pd.concat(list_df, ignore_index=True)
print("setelah digabungkan:")
df_combined.tail()

setelah digabungkan:


,node,location,weather,datetime,h2s,so2,hum,temp,windspeed,elevation,latitude,longitude,hour,minute,minute_of_day,h2s_diff,so2_diff,gas_ratio_so2_h2s
2890,6,Tangga Masuk Pengunjung,Hujan Kabut,2025-12-22 16:43:24,7.0,153.5,100.0,14.71,0.36,2200.0,-7.166833,107.40411,16,43,1003,3.0,0.0,21.928568
2891,6,Tangga Masuk Pengunjung,Hujan Kabut,2025-12-22 16:43:26,7.0,153.5,100.0,14.71,0.72,2200.0,-7.166833,107.40411,16,43,1003,0.0,0.0,21.928568
2892,6,Tangga Masuk Pengunjung,Hujan Kabut,2025-12-22 16:43:28,7.0,153.5,100.0,14.71,0.72,2200.0,-7.166833,107.40411,16,43,1003,0.0,0.0,21.928568
2893,6,Tangga Masuk Pengunjung,Hujan Kabut,2025-12-22 16:43:30,7.0,153.5,100.0,14.71,0.72,2200.0,-7.166833,107.40411,16,43,1003,0.0,0.0,21.928568
2894,6,Tangga Masuk Pengunjung,Hujan Kabut,2025-12-22 16:43:32,7.0,153.5,100.0,14.71,0.36,2200.0,-7.166833,107.40411,16,43,1003,0.0,0.0,21.928568


In [7]:
df_combined.to_csv(project_root / 'data' / 'processed' / 'node_combined_final.csv', index=False)